1.	Выполните сохранение монохромного изображения в виде текстового или бинарного файла.
2.	Реализуйте алгоритм вейвлет-преобразования Хаара для изображения.
3.	Выполните квантование высокочастотных компонент (прим., количество квантов  = 4).
4.	Сохраните получившийся массив значений  в текстовый или бинарный файл в порядке LL, LH, HL, HH вейвлет-преобразования Хафа. Компоненты LH, HL, HH храните в виде пар (значение, количество повторений).

Сравните объем памяти, занимаемый исходным изображением (попиксельное хранение), и изображение, полученным после преобразования Хафа и сжатием длин серий.


In [16]:
import numpy as np
import cv2
import os

In [17]:
def save_monochrome_binary(image, filename):
    with open(filename, 'wb') as f:
        f.write(image.shape[0].to_bytes(4, 'big'))
        f.write(image.shape[1].to_bytes(4, 'big'))
        f.write(image.tobytes())

def haar_transform(img):
    h, w = img.shape
    h = h - h % 2
    w = w - w % 2
    img = img[:h, :w].astype(np.float64)
    
    result = np.zeros_like(img)
    
    # Преобразование по строкам
    for i in range(h):
        row = img[i, :w]
        avg = (row[::2] + row[1::2]) / 2
        diff = (row[::2] - row[1::2]) / 2
        result[i, :w//2] = avg
        result[i, w//2:w] = diff
    
    # Преобразование по столбцам
    for j in range(w):
        col = result[:h, j]
        avg = (col[::2] + col[1::2]) / 2
        diff = (col[::2] - col[1::2]) / 2
        result[:h//2, j] = avg
        result[h//2:h, j] = diff
    
    return result

def quantize(component, levels=4):
    min_val, max_val = np.min(component), np.max(component)
    if min_val == max_val:
        return np.zeros_like(component, dtype=int)
    
    bins = np.linspace(min_val, max_val, levels + 1)
    quantized = np.digitize(component, bins) - 1
    quantized = np.clip(quantized, 0, levels - 1)
    return quantized

def rle_encode(arr):
    flat = arr.ravel()
    if len(flat) == 0:
        return np.array([], dtype=int)
    
    change_idx = np.where(flat[1:] != flat[:-1])[0] + 1
    splits = np.split(flat, change_idx)
    res = np.array([(s[0], len(s)) for s in splits], dtype=int)
    return res

def save_haar_components(filename, ll, lh, hl, hh):
    with open(filename, 'w') as f:
        # Сохраняем LL компоненту
        f.write(f"LL {ll.shape[0]} {ll.shape[1]}\n")
        np.savetxt(f, ll, fmt='%.6f')
        
        # Кодируем и сохраняем высокочастотные компоненты
        for component, name in [(lh, "LH"), (hl, "HL"), (hh, "HH")]:
            rle_data = rle_encode(component)
            f.write(f"{name} {len(rle_data)}\n")
            for value, count in rle_data:
                f.write(f"{value} {count}\n")

In [18]:
# 1. Загрузка и сохранение монохромного изображения
image = cv2.imread('mono_image.jpg', cv2.IMREAD_GRAYSCALE)
if image is None:
    print("Ошибка: не удалось загрузить изображение")
    
save_monochrome_binary(image, 'monochrome_binary.dat')
print("Монохромное изображение сохранено в бинарном формате")
    
# 2. Вейвлет-преобразование Хаара
transformed = haar_transform(image)
h, w = transformed.shape
    
# Разделение на компоненты
ll = transformed[:h//2, :w//2]
lh = transformed[:h//2, w//2:]
hl = transformed[h//2:, :w//2]
hh = transformed[h//2:, w//2:]
    
# 3. Квантование высокочастотных компонент
lh_quantized = quantize(lh, 4)
hl_quantized = quantize(hl, 4)
hh_quantized = quantize(hh, 4)
    
# 4. Сохранение результатов
save_haar_components('haar_transform.txt', ll, lh_quantized, hl_quantized, hh_quantized)
print("Результаты вейвлет-преобразования сохранены")
    
# Сравнение размеров файлов
original_size = os.path.getsize('monochrome_binary.dat')
compressed_size = os.path.getsize('haar_transform.txt')
    
print(f"\nСравнение размеров файлов:")
print(f"Исходный файл: {original_size} байт")
print(f"Сжатый файл: {compressed_size} байт")
print(f"Коэффициент сжатия: {original_size/compressed_size:.2f}")

Монохромное изображение сохранено в бинарном формате
Результаты вейвлет-преобразования сохранены

Сравнение размеров файлов:
Исходный файл: 99128 байт
Сжатый файл: 327846 байт
Коэффициент сжатия: 0.30
